# 10 · Candidate partition workflow

Run and compare comfort-category candidates with GroupKFold ML evaluation.

In [ ]:
from pathlib import Path
import pandas as pd

from candidate_partition_workflow import run_candidate_workflow, plot_confusiont
from utils_ml_adv import analyze_binary_detector

## Load prepared dataframe

In [ ]:
# Option A: if still in memory from notebook 00
# df_prepared


# Option B: load from saved CSV

prepared_path = Path("saved_df/sens_prepared_df.csv")
df_prepared = pd.read_csv(prepared_path)
# print(df_prepared.shape)
df_prepared.columns


## Run candidate benchmark

In [ ]:
cand = run_candidate_workflow(
    df_prepared,
    include_xgb=False,
    save_prefix="candidate"
)

cand.head(20)

## Plot confusion matrix for a selected candidate/model

In [ ]:

cm = plot_confusiont(
    df_prepared,
    outcome_col="sens7",
    feature_set_name="full_context",
    model_name="logreg",
    include_xgb=False,
    output_path="cm_sens7_logreg.png",
)
cm["cm_counts"]
cm["cm_percent"]

cm = plot_confusiont(
    df_prepared,
    outcome_col="sens3_option1",
    feature_set_name="full_context",
    model_name="logreg",
    include_xgb=False,
    output_path="cm_sens3_option1_logreg.png",
)
cm


cm = plot_confusiont(
    df_prepared,
    outcome_col="sens3_option2",
    feature_set_name="full_context",
    model_name="logreg",
    include_xgb=False,
    output_path="cm_sens3_option3_logreg.png",
)
cm

# Mirem de fer-ho ordinal i també mirem-ho de manera binària (categoria vs rest)

In [ ]:
# =========================================================
# FINAL ML TOUCHES:
# - ordinal comfort3_option1
# - binary detectors at different discomfort cutpoints
# - confusion matrices with counts + row %
# - threshold tuning for rare states
# =========================================================




import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from candidate_partition_workflow import *
from pathlib import Path
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    confusion_matrix,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression


# ---------------------------------------------------------
# 0. Output folders
# ---------------------------------------------------------
FIG_DIR = Path("figures_sens")
CSV_DIR = Path("csvs_sens")
FIG_DIR.mkdir(exist_ok=True)
CSV_DIR.mkdir(exist_ok=True)


# ---------------------------------------------------------
# 1. Helper: robust OHE
# ---------------------------------------------------------
def make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


# ---------------------------------------------------------
# 2. Helper: choose feature columns from DEFAULT_FEATURES
# ---------------------------------------------------------
def get_feature_cols(df, feature_set_name="full_context"):
    cols = [c for c in DEFAULT_FEATURES[feature_set_name] if c in df.columns]
    if len(cols) == 0:
        raise ValueError(f"No columns found for feature_set_name={feature_set_name}")
    return cols


# ---------------------------------------------------------
# 3. Generic preprocessing
# ---------------------------------------------------------
def make_preprocessor_from_X(X):
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = [c for c in X.columns if c not in num_cols]

    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ohe", make_ohe()),
    ])

    pre = ColumnTransformer([
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols),
    ])

    return pre


# ---------------------------------------------------------
# 4. Plot confusion matrix:
#    colors = row percentages
#    text = counts + row percentages
# ---------------------------------------------------------
def plot_cm_counts_rowpct(
    cm_counts,
    cm_rowpct,
    labels,
    title="",
    output_path=None,
    show_support=None,
    cmap="Blues",
):
    labels = list(labels)

    if show_support is not None:
        display_labels = [f"{lab}\n(n={show_support.get(lab, 0)})" for lab in labels]
    else:
        display_labels = labels

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm_rowpct, cmap=cmap, vmin=0, vmax=1)

    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels(display_labels, rotation=45, ha="right")
    ax.set_yticklabels(display_labels)

    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_title(title)

    for i in range(cm_counts.shape[0]):
        for j in range(cm_counts.shape[1]):
            count = int(cm_counts[i, j])
            pct = cm_rowpct[i, j]
            txt = f"{count}\n({pct:.2f})"
            text_color = "white" if pct > 0.45 else "#123b7a"
            ax.text(j, i, txt, ha="center", va="center", color=text_color, fontsize=10)

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Row-normalized proportion")

    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=220, bbox_inches="tight")
    plt.show()


# ---------------------------------------------------------
# 5. Ordinal target creation: comfort3_option1
#    comfortable-neutral < uncomfortable < very uncomfortable
# ---------------------------------------------------------
def build_ordinal_comfort3_option1(df, source_col="thermal_comfort"):
    out = df.copy()

    def recode(x):
        if pd.isna(x):
            return np.nan
        if x in ["Very comfortable", "Comfortable", "Slightly comfortable", "Neutral"]:
            return "comfortable-neutral"
        if x in ["Slightly uncomfortable", "Uncomfortable"]:
            return "uncomfortable"
        if x == "Very uncomfortable":
            return "very uncomfortable"
        return np.nan

    out["comfort3_option1"] = out[source_col].map(recode)
   
    return out


# ---------------------------------------------------------
# 6. Binary cutpoint targets on original 7-level comfort
# ---------------------------------------------------------
def add_binary_discomfort_targets(df, source_col="thermal_comfort"):
    out = df.copy()

    out["is_very_uncomfortable"] = (
        out[source_col] == "Very uncomfortable"
    ).astype("float")

    out["is_uncomfortable_or_worse"] = (
        out[source_col].isin(["Uncomfortable", "Very uncomfortable"])
    ).astype("float")

    out["is_slightly_uncomfortable_or_worse"] = (
        out[source_col].isin(["Slightly uncomfortable", "Uncomfortable", "Very uncomfortable"])
    ).astype("float") 

    # keep NaN where original is NaN
    na_mask = out[source_col].isna()
    out.loc[na_mask, "is_very_uncomfortable"] = np.nan
    out.loc[na_mask, "is_uncomfortable_or_worse"] = np.nan
    out.loc[na_mask, "is_slightly_uncomfortable_or_worse"] = np.nan

    return out


# ---------------------------------------------------------
# 7. Threshold tuning for binary detectors
#    Default: maximize F1 
# ---------------------------------------------------------
def choose_threshold_from_oof(
    y_true,
    y_score,
    thresholds=None,
    objective="f1",   # "f1", "f2", "balanced_accuracy", "recall", "precision"
):
    if thresholds is None:
        thresholds = np.linspace(0.05, 0.95, 37)

    rows = []

    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)

    for thr in thresholds:
        y_pred = (y_score >= thr).astype(int)

        prec = precision_score(y_true, y_pred, zero_division=0)
        rec = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)

        # F2 per si encara la vols tenir disponible
        if prec == 0 and rec == 0:
            f2 = 0.0
        else:
            beta2 = 2.0 ** 2
            f2 = (1 + beta2) * prec * rec / (beta2 * prec + rec)

        bal_acc = balanced_accuracy_score(y_true, y_pred)

        rows.append({
            "threshold": thr,
            "precision": prec,
            "recall": rec,
            "f1": f1,
            "f2": f2,
            "balanced_accuracy": bal_acc,
        })

    th_df = pd.DataFrame(rows)

    if objective not in th_df.columns:
        raise ValueError(f"objective must be one of {list(th_df.columns)}")

    th_df = th_df.sort_values(
        [objective, "balanced_accuracy", "recall", "precision"],
        ascending=[False, False, False, False]
    ).reset_index(drop=True)

    best = th_df.iloc[0].to_dict()
    return best, th_df


# ---------------------------------------------------------
# 8. Ordinal model:
#    regress on 0/1/2 and round back to nearest state
# ---------------------------------------------------------
def run_grouped_ordinal_model(
    df,
    outcome_col="comfort3_option1",
    feature_set_name="full_context",
    group_col="ID",
    n_splits=5,
    model_name="rf_ord",
    label_order=None,
    order_map=None,
    output_path=None,
):
    """
    Ordinal-style grouped CV model using a regressor on coded target values.

    Important:
    - If `order_map` is None, uses uniform spacing from label_order:
        comfortable-neutral -> 0
        uncomfortable -> 1
        very uncomfortable -> 2
    - If `order_map` is provided, it can use non-uniform spacing, e.g.:
        {"comfortable-neutral": 0, "uncomfortable": 1, "very uncomfortable": 3}
    - Predictions are decoded to the nearest allowed code, NOT by plain rounding.
    """

    if label_order is None:
        label_order = ["comfortable-neutral", "uncomfortable", "very uncomfortable"]

    # Default = equally spaced ordinal coding
    if order_map is None:
        order_map = {lab: i for i, lab in enumerate(label_order)}

    # Safety checks
    missing = [lab for lab in label_order if lab not in order_map]
    if len(missing) > 0:
        raise ValueError(f"order_map is missing labels: {missing}")

    # Allowed numeric codes in the same semantic order as label_order
    code_values = np.array([order_map[lab] for lab in label_order], dtype=float)
    inv_map = {order_map[lab]: lab for lab in label_order}

    cols = get_feature_cols(df, feature_set_name)
    tmp = df[[outcome_col, group_col] + cols].dropna(subset=[outcome_col, group_col]).copy()

    X = tmp[cols].copy()
    y_lab = tmp[outcome_col].copy()
    y = y_lab.map(order_map).astype(float)
    groups = tmp[group_col].copy()

    pre = make_preprocessor_from_X(X)

    if model_name == "rf_ord":
        model = RandomForestRegressor(
            n_estimators=500,
            random_state=42,
            min_samples_leaf=5,
            n_jobs=-1,
        )
    else:
        raise ValueError("Supported ordinal models: 'rf_ord'")

    pipe = Pipeline([
        ("pre", pre),
        ("model", model),
    ])

    gkf = GroupKFold(n_splits=n_splits)
    y_true_all, y_pred_all, y_pred_cont_all = [], [], []

    def decode_to_nearest_code(pred_cont_values, allowed_codes):
        pred_cont_values = np.asarray(pred_cont_values, dtype=float)
        out = []
        for x in pred_cont_values:
            nearest = allowed_codes[np.argmin(np.abs(allowed_codes - x))]
            out.append(float(nearest))
        return np.array(out, dtype=float)

    for tr, te in gkf.split(X, y, groups):
        fitted = clone(pipe)
        fitted.fit(X.iloc[tr], y.iloc[tr])

        pred_cont = fitted.predict(X.iloc[te])

        # IMPORTANT:
        # map each continuous prediction to the nearest allowed code
        pred_ord = decode_to_nearest_code(pred_cont, code_values)

        y_true_all.extend(y.iloc[te].tolist())
        y_pred_all.extend(pred_ord.tolist())
        y_pred_cont_all.extend(pred_cont.tolist())

    # Decode numeric codes back to semantic labels
    y_true_lab = pd.Series([inv_map[float(v)] for v in y_true_all], name="y_true")
    y_pred_lab = pd.Series([inv_map[float(v)] for v in y_pred_all], name="y_pred")

    # Confusion matrices
    cm_counts = confusion_matrix(y_true_lab, y_pred_lab, labels=label_order, normalize=None)
    cm_rowpct = confusion_matrix(y_true_lab, y_pred_lab, labels=label_order, normalize="true")

    support = y_true_lab.value_counts()

    plot_cm_counts_rowpct(
        cm_counts=cm_counts,
        cm_rowpct=cm_rowpct,
        labels=label_order,
        show_support=support,
        title=f"{outcome_col} | ordinal {model_name} | {feature_set_name} | codes={order_map}",
        output_path=output_path,
    )

    # Ordinal metrics
    yt = np.asarray(y_true_all, dtype=float)
    yp = np.asarray(y_pred_all, dtype=float)

    # For balanced accuracy we need compact class ids 0..K-1
    compact_map = {lab: i for i, lab in enumerate(label_order)}
    yt_compact = y_true_lab.map(compact_map).to_numpy()
    yp_compact = y_pred_lab.map(compact_map).to_numpy()

    metrics = {
        "accuracy": np.mean(y_true_lab.to_numpy() == y_pred_lab.to_numpy()),
        "balanced_accuracy": balanced_accuracy_score(yt_compact, yp_compact),
        "mae_ordinal": np.mean(np.abs(yt - yp)),
        "mse_ordinal": np.mean((yt - yp) ** 2),
        # "far error" = jump larger than the smallest adjacent gap
        "far_error_rate": np.mean(np.abs(yt - yp) >= (np.min(np.diff(np.sort(code_values))) + 1e-12)),
    }

    return {
        "cm_counts": pd.DataFrame(cm_counts, index=label_order, columns=label_order),
        "cm_rowpct": pd.DataFrame(cm_rowpct, index=label_order, columns=label_order),
        "metrics": pd.DataFrame([metrics]),
        "oof": pd.DataFrame({
            "y_true": y_true_lab,
            "y_pred": y_pred_lab,
            "y_true_ord": yt,
            "y_pred_ord": yp,
            "y_pred_cont": y_pred_cont_all,
        }),
    }

# ---------------------------------------------------------
# 9. Binary detector with threshold tuning
# ---------------------------------------------------------
def run_grouped_binary_detector(
    df,
    target_col,
    feature_set_name="full_context",
    group_col="ID",
    n_splits=5,
    model_name="rf",
    threshold="auto",
    threshold_objective="f1",
    output_path=None,
):
    cols = get_feature_cols(df, feature_set_name)

    # ---- metadata columns to preserve in OOF output ----
    required_meta_cols = ["row_id","ID"]
    optional_meta_cols = ["space_code", "walk_id", "stop_idx"]

    missing_required = [c for c in required_meta_cols if c not in df.columns]
    if missing_required:
        raise ValueError(
            f"Missing required metadata columns in df: {missing_required}. "
            "Create them before calling run_grouped_binary_detector."
        )

    meta_cols = required_meta_cols + [c for c in optional_meta_cols if c in df.columns]

    # safety check: metadata / target / grouping column must NOT enter the feature matrix
    forbidden_in_features = set(meta_cols + [target_col, group_col])
    overlap = [c for c in cols if c in forbidden_in_features]
    if overlap:
        raise ValueError(f"These non-feature columns are present in feature set: {overlap}")

    # avoid duplicated columns (important when group_col or others overlap)
    base_cols = meta_cols + [target_col, group_col] + cols
    use_cols = list(dict.fromkeys(base_cols))

    tmp = df.loc[:, use_cols].dropna(subset=[target_col, group_col]).copy()

    # make sure row_id is preserved as a plain column with stable type
    tmp["row_id"] = tmp["row_id"].astype(int)

    X = tmp[cols].copy()
    y = tmp[target_col].astype(int).copy()
    groups = tmp[group_col].copy()

    pre = make_preprocessor_from_X(X)

    if model_name == "rf":
        model = RandomForestClassifier(
            n_estimators=500,
            random_state=42,
            min_samples_leaf=3,
            class_weight="balanced_subsample",
            n_jobs=-1,
        )
    elif model_name == "logreg":
        model = LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
        )
    else:
        raise ValueError("Supported binary models: 'rf', 'logreg'")

    pipe = Pipeline([
        ("pre", pre),
        ("model", model),
    ])

    gkf = GroupKFold(n_splits=n_splits)

    # collect fold-wise OOF rows instead of only raw arrays
    oof_parts = []

    for fold, (tr, te) in enumerate(gkf.split(X, y, groups), start=1):
        fitted = clone(pipe)
        fitted.fit(X.iloc[tr], y.iloc[tr])

        if hasattr(fitted, "predict_proba"):
            score = fitted.predict_proba(X.iloc[te])[:, 1]
        else:
            score = fitted.predict(X.iloc[te]).astype(float)

        # use .loc with the actual index labels of the held-out rows
        test_idx = X.iloc[te].index
        part = tmp.loc[test_idx, meta_cols].copy()

        part["fold"] = fold
        part["y_true"] = y.loc[test_idx].to_numpy().astype(int)
        part["y_score"] = np.asarray(score, dtype=float)

        oof_parts.append(part)

    oof = pd.concat(oof_parts, axis=0, ignore_index=True)

    # sorting by row_id gives stable original-row ordering
    oof = oof.sort_values("row_id").reset_index(drop=True)

    y_true_all = oof["y_true"].to_numpy().astype(int)
    y_score_all = oof["y_score"].to_numpy().astype(float)

    # threshold selection
    if threshold == "auto":
        best_thr, threshold_table = choose_threshold_from_oof(
            y_true_all,
            y_score_all,
            objective=threshold_objective
        )
        thr = best_thr["threshold"]
    elif isinstance(threshold, (float, int)):
        thr = float(threshold)
        _, threshold_table = choose_threshold_from_oof(
            y_true_all,
            y_score_all,
            objective=threshold_objective
        )
    else:
        raise ValueError("threshold must be 'auto' or a numeric value")

    oof["y_pred"] = (oof["y_score"] >= thr).astype(int)
    y_pred_all = oof["y_pred"].to_numpy().astype(int)

    labels = [0, 1]
    label_order = ["negative", "positive"]
    label_names = {
        "is_very_uncomfortable": ["not very uncomfortable", "very uncomfortable"],
        "is_uncomfortable_or_worse": ["not U-or-worse", "U or very uncomfortable"],
        "is_slightly_uncomfortable_or_worse": ["not SU-or-worse", "SU/U/VU"],
    }
    label_order = label_names.get(target_col, label_order)

    cm_counts = confusion_matrix(y_true_all, y_pred_all, labels=labels, normalize=None)
    cm_rowpct = confusion_matrix(y_true_all, y_pred_all, labels=labels, normalize="true")

    support = pd.Series(y_true_all).map({0: label_order[0], 1: label_order[1]}).value_counts()

    plot_cm_counts_rowpct(
        cm_counts=cm_counts,
        cm_rowpct=cm_rowpct,
        labels=label_order,
        show_support=support,
        title=f"{target_col} | {model_name} | {feature_set_name} | thr={thr:.2f}",
        output_path=output_path,
    )

    metrics = {
        "threshold_used": thr,
        "prevalence_positive": float(np.mean(y_true_all)),
        "balanced_accuracy": balanced_accuracy_score(y_true_all, y_pred_all),
        "precision_positive": precision_score(y_true_all, y_pred_all, zero_division=0),
        "recall_positive": recall_score(y_true_all, y_pred_all, zero_division=0),
        "f1_positive": f1_score(y_true_all, y_pred_all, zero_division=0),
        "average_precision": average_precision_score(y_true_all, y_score_all),
    }

    try:
        metrics["roc_auc"] = roc_auc_score(y_true_all, y_score_all)
    except Exception:
        metrics["roc_auc"] = np.nan

    return {
        "cm_counts": pd.DataFrame(cm_counts, index=label_order, columns=label_order),
        "cm_rowpct": pd.DataFrame(cm_rowpct, index=label_order, columns=label_order),
        "metrics": pd.DataFrame([metrics]),
        "threshold_table": threshold_table,
        "oof": oof,
    }

# ---------------------------------------------------------
# 10. Example workflow
# ---------------------------------------------------------

# 10.1 Make sure the grouped outcomes exist
#df = build_ordinal_comfort3_option1(df_prepared, source_col="thermal_comfort")
#df = add_binary_discomfort_targets(df_prepared, source_col="thermal_comfort")
df = df_prepared.copy()


# 10.3 Binary detector 1: Very uncomfortable vs rest
vu_res = run_grouped_binary_detector(
    df,
    target_col="is_very_hot",
    feature_set_name="full_context",
    group_col="ID",
    n_splits=5,
    model_name="logreg",
    threshold="auto",
    threshold_objective="f1",
    output_path=str(FIG_DIR / "binary_is_very_hot_logreg_f1.png"),
)

display(vu_res["metrics"])
display(vu_res["cm_counts"])
display(vu_res["cm_rowpct"])

vu_res["metrics"].to_csv(CSV_DIR / "binary_is_very_hot_metrics.csv", index=False)
vu_res["cm_counts"].to_csv(CSV_DIR / "binary_is_very_hot_cm_counts.csv")
vu_res["cm_rowpct"].to_csv(CSV_DIR / "binary_is_very_hot_cm_rowpct.csv")
vu_res["threshold_table"].to_csv(CSV_DIR / "binary_is_very_hot_thresholds.csv", index=False)
vu_res["oof"].to_csv(CSV_DIR / "very_hot_vs_rest_oof.csv", index=False)


# 10.4 Binary detector 2: Uncomfortable or worse vs rest
u_res = run_grouped_binary_detector(
    df,
    target_col="is_hot_or_very_hot",
    feature_set_name="full_context",
    group_col="ID",
    n_splits=5,
    model_name="logreg",
    threshold="auto",
    threshold_objective="f1",
    output_path=str(FIG_DIR / "binary_is_hot_or_very_hot_logreg_f1.png"),
)

display(u_res["metrics"])
display(u_res["cm_counts"])
display(u_res["cm_rowpct"])

u_res["metrics"].to_csv(CSV_DIR / "binary_is_hot_or_very_hot_metrics.csv", index=False)
u_res["cm_counts"].to_csv(CSV_DIR / "binary_is_hot_or_very_hot_cm_counts.csv")
u_res["cm_rowpct"].to_csv(CSV_DIR / "binary_is_hot_or_very_hot_cm_rowpct.csv")
u_res["threshold_table"].to_csv(CSV_DIR / "binary_is_hot_or_very_hot_thresholds.csv", index=False)
u_res["oof"].to_csv(CSV_DIR / "hot_vs_rest_oof.csv", index=False)


# 10.5 Binary detector 3: Slightly uncomfortable or worse vs rest
su_res = run_grouped_binary_detector(
    df,
    target_col="is_warm_or_hot",
    feature_set_name="full_context",
    group_col="ID",
    n_splits=5,
    model_name="logreg",
    threshold="auto",
    threshold_objective="f1",
    output_path=str(FIG_DIR / "binary_is_warm_or_hot_logreg_f1.png"),
)

display(su_res["metrics"])
display(su_res["cm_counts"])
display(su_res["cm_rowpct"])

su_res["metrics"].to_csv(CSV_DIR / "binary_is_warm_or_hot_metrics.csv", index=False)
su_res["cm_counts"].to_csv(CSV_DIR / "binary_is_warm_or_hot_cm_counts.csv")
su_res["cm_rowpct"].to_csv(CSV_DIR / "binary_is_warm_or_hot_cm_rowpct.csv")
su_res["threshold_table"].to_csv(CSV_DIR / "binary_is_warm_or_hot_thresholds.csv", index=False)
su_res["oof"].to_csv(CSV_DIR / "warm-hot_vs_rest_oof.csv", index=False)







# Merging dataframes to have the binary information

In [ ]:
oof_vu = pd.read_csv(CSV_DIR / "very_hot_vs_rest_oof.csv")
oof_uw = pd.read_csv(CSV_DIR / "hot_vs_rest_oof.csv")
oof_suw = pd.read_csv(CSV_DIR / "warm-hot_vs_rest_oof.csv")

oof_vu_small = oof_vu[["row_id", "fold", "y_true", "y_score", "y_pred"]].rename(
    columns={
        "fold": "vu_fold",
        "y_true": "vu_true",
        "y_score": "vu_score",
        "y_pred": "vu_pred",
    }
)

oof_uw_small = oof_uw[["row_id", "fold", "y_true", "y_score", "y_pred"]].rename(
    columns={
        "fold": "uw_fold",
        "y_true": "uw_true",
        "y_score": "uw_score",
        "y_pred": "uw_pred",
    }
)

oof_suw_small = oof_suw[["row_id", "fold", "y_true", "y_score", "y_pred"]].rename(
    columns={
        "fold": "suw_fold",
        "y_true": "suw_true",
        "y_score": "suw_score",
        "y_pred": "suw_pred",
    }
)

df_final = df.merge(oof_vu_small, on="row_id", how="left")
df_final = df_final.merge(oof_uw_small, on="row_id", how="left")
df_final = df_final.merge(oof_suw_small, on="row_id", how="left")

df_final = df_final.sort_values(["ID", "stop_idx", "row_id"]).reset_index(drop=True)

print("Rows original core:", len(df))
print("Rows final:", len(df_final))
print("Unique row_id final:", df_final["row_id"].nunique())
print("Duplicated row_id final:", df_final["row_id"].duplicated().sum())
print(df_final[["vu_score", "uw_score"]].isna().mean())
mask_vu = df_final["vu_true"].notna()
print("VU truth agreement:",
      (df_final.loc[mask_vu, "vu_true"].astype(int) ==
       df_final.loc[mask_vu, "is_very_uncomfortable"].astype(int)).mean())

mask_uw = df_final["uw_true"].notna()


df_final.to_csv("saved_df/df_dynamics_with_oof_sensation.csv", index=False)

# I mirem les variables més rellevants per al descomfort binari

In [ ]:

df_prepared = add_binary_discomfort_targets(df_prepared, source_col="thermal_comfort")

# 1) Very uncomfortable vs rest
vu_analysis = analyze_binary_detector(
    df_prepared,
    target_col="is_very_hot",
    feature_set_name="full_context",
    group_col="ID",
    n_splits=5,
    model_name="logreg",
    threshold="auto",
    threshold_objective="f1",
    top_n_cases=40,
)

# 2) Uncomfortable or worse vs rest
uw_analysis = analyze_binary_detector(
    df_prepared,
    target_col="is_hot_or_very_hot",
    feature_set_name="full_context",
    group_col="ID",
    n_splits=5,
    model_name="logreg",
    threshold="auto",
    threshold_objective="f1",
    top_n_cases=40,
)

su_analysis = analyze_binary_detector(
    df_prepared,
    target_col="is_warm_or_hot",
    feature_set_name="full_context",
    group_col="ID",
    n_splits=5,
    model_name="logreg",
    threshold="auto",
    threshold_objective="f1",
    top_n_cases=40,
)




display(vu_analysis["metrics"])
display(vu_analysis["importance_summary"].head(15))
display(uw_analysis["metrics"])
display(uw_analysis["importance_summary"].head(15))
display(su_analysis["metrics"])
display(su_analysis["importance_summary"].head(15))

# Save key outputs
for prefix, res in [("vu", vu_analysis), ("uw", uw_analysis), ("su", su_analysis)]:
    res["metrics"].to_csv(CSV_DIR / f"{prefix}_binary_metrics.csv", index=False)
    res["threshold_table"].to_csv(CSV_DIR / f"{prefix}_threshold_table.csv", index=False)
    res["importance_summary"].to_csv(CSV_DIR / f"{prefix}_importance_summary.csv", index=False)
    res["numeric_case_summary"].to_csv(CSV_DIR / f"{prefix}_numeric_case_summary.csv", index=False)
    res["categorical_case_summary"].to_csv(CSV_DIR / f"{prefix}_categorical_case_summary.csv", index=False)
    res["top_predicted_positive"].to_csv(CSV_DIR / f"{prefix}_top_predicted_positive.csv", index=False)
    res["top_true_positives"].to_csv(CSV_DIR / f"{prefix}_top_true_positives.csv", index=False)
    res["false_positives"].to_csv(CSV_DIR / f"{prefix}_false_positives.csv", index=False)
    res["false_negatives"].to_csv(CSV_DIR / f"{prefix}_false_negatives.csv", index=False)